In [1]:
# Laden der Daten von Kaggle auf Colab
import os
import kagglehub
import pandas as pd

download_path = kagglehub.dataset_download("gokulraja84/emails-dataset-for-spam-detection")
csv_file_path = os.path.join(download_path, 'Emails.csv')
raw_df = pd.read_csv(csv_file_path)
#display(raw_df)

C:\Users\Administrator\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


100%|█████████████████████████████████████████████████████████████████████████████| 2.86M/2.86M [00:00<00:00, 3.78MB/s]

Extracting files...


In [2]:
# doppelte Vorkommen entfernen
raw_unique_df = raw_df.drop_duplicates(subset=['text'], keep='first').copy()
print(f"Größe raw_df: {len(raw_df)} Zeilen")
print(f"Größe raw_unique_df: {len(raw_unique_df)} Zeilen")

Größe raw_df: 5728 Zeilen
Größe raw_unique_df: 5695 Zeilen


In [3]:
# Konvertiere label zu {0, 1}
all_df = raw_unique_df.copy()
all_df['label'] = (all_df['spam']).astype(int)
all_df = all_df.drop('spam', axis=1)
#display(all_df)

In [4]:
# Aufteilung in train_df und validation_df
from sklearn.model_selection import train_test_split

RANDOM_SEED = 42
SHARE_TEST = 0.2
train_df, validation_df = train_test_split(
    all_df,
    test_size=SHARE_TEST,    # 20% für validation_df
    random_state=RANDOM_SEED,  # Fester Random-Seed für Reproduzierbarkeit
    stratify=all_df['label'] # Stratifizieren nach Label, um eine gleiche Verteilung zu gewährleisten
)
train_df.reset_index(drop=True, inplace=True) #lässt den Index wieder von 1 starten
validation_df.reset_index(drop=True, inplace=True)
print(f"Größe von train_df: {len(train_df)} Zeilen")
print(f"Größe von validation_df: {len(validation_df)} Zeilen")

Größe von train_df: 4556 Zeilen
Größe von validation_df: 1139 Zeilen


In [5]:
# Trainingsdaten ansehen
display(train_df)

,text,label
0,"Subject: energy book hi grant , hope all is ...",0
1,"Subject: re : enron contact info christie , ...",0
2,Subject: improved process for engaging tempora...,0
3,Subject: . jif .,1
4,"Subject: parameter estimation vince , i have...",0
...,...,...
4551,"Subject: a paper of mine vince , i have writ...",0
4552,Subject: re : aiesec polska - eurolds 2000 dr...,0
4553,"Subject: localized software , all languages av...",1
4554,"Subject: happy thanksgiving ! hello , vince !...",0


In [6]:
# Validation-Daten ansehen
display(validation_df)

,text,label
0,"Subject: bachelier finance society congress , ...",0
1,"Subject: congratulations vince , congratulat...",0
2,"Subject: re : resume , thanks a lot , and ju...",0
3,"Subject: re : recommendation letter vincent ,...",0
4,Subject: pac enrollment last year the enron p...,0
...,...,...
1134,Subject: re : managing energy price risk - 2 n...,0
1135,Subject: re : meeting on the 20 th of march f...,0
1136,Subject: faculty information sheet mr . kamin...,0
1137,Subject: selling travel in today ' s economy ...,1


In [64]:
# Imports

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import matthews_corrcoef
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neural_network import MLPClassifier

In [65]:
# TF-Idf Vectorizer definiert
# Vectorizer wurde für beide Modelle verwendet (Ich hoffe der Unterschied im Modell entspricht der Unterscheidung ausreichend)

tfidf_vectorizer = TfidfVectorizer(
    lowercase=True,        # alles klein schreiben
    stop_words='english',  # häufige englische Wörter entfernen
    max_df=0.95,           # sehr häufige Wörter ignorieren
    min_df=2               # extrem seltene Wörter ignorieren
)

X_train = tfidf_vectorizer.fit_transform(train_df['text'])
y_train = train_df['label']
X_val = tfidf_vectorizer.transform(validation_df['text'])
y_val = validation_df['label']

# zur dimensionalen Übersicht in train und validation
print(X_train.shape)
print(X_val.shape)


(4556, 17659)
(1139, 17659)


In [66]:
# Logistisches Modell - Verfahren 1
# folgende Anpassungen wurden vorgenommen:
# - max_iter=1000: stellt sicher, dass der Optimierer vollständig konvergiert
# - class_weight='balanced': gleicht die Klassenverteilung aus (Spam/nicht spam) und verbessert die Klassifikation bei ungleich verteilten Labels
# - random_state=42: feste Zufallswerte für Reproduzierbarkeit


logreg_model = LogisticRegression(
    max_iter=1000,           
    class_weight='balanced', 
    random_state=42
)

logreg_model.fit(X_train, y_train);


In [67]:
# Prediction Zelle + MCC-Auswertung:

train_predictions = logreg_model.predict(X_train)
val_predictions = logreg_model.predict(X_val)

mcc_train = matthews_corrcoef(y_train, train_predictions)
mcc_val = matthews_corrcoef(y_val, val_predictions)

print(f"MCC Logistic Regression (train_df): {mcc_train:.3f}")
print(f"MCC Logistic Regression (validation_df): {mcc_val:.3f}")


MCC Logistic Regression (train_df): 0.985
MCC Logistic Regression (validation_df): 0.976


In [70]:
# Neuronales Netz - Verfahren 2
# -> Validation etwas schlechter als bei Logistic Regression, deswegen Verfahren 2
# folgende Anpassungen wurden vorgenommen:
# - hidden_layer_sizes=(50,): Hidden Layer mit 50 Neuronen
# - activation='relu': ReLU als Aktivierungsfunktion
# - solver='adam': Optimierer für das Training
# - max_iter=200: maximale Iterationen
# - early_stopping=True: stoppt Training bei stagnierender Validation
# - n_iter_no_change=10: Anzahl Iterationen ohne Verbesserung für Early Stopping
# - random_state=42: feste Zufallswerte für Reproduzierbarkeit

mlp_model = MLPClassifier(
    hidden_layer_sizes=(50,),   # 1 versteckte Schicht mit 50 Neuronen
    activation='relu',          
    solver='adam',              
    max_iter=200,              
    random_state=42,
    early_stopping=True,        
    n_iter_no_change=10,
    verbose=False
)

mlp_model.fit(X_train, y_train);

In [71]:
# Prediction Zelle + MCC-Auswertung:

train_pred_mlp = mlp_model.predict(X_train)
val_pred_mlp = mlp_model.predict(X_val)

mcc_train_mlp = matthews_corrcoef(y_train, train_pred_mlp)
mcc_val_mlp = matthews_corrcoef(y_val, val_pred_mlp)

print(f"MCC MLP (train_df): {mcc_train_mlp:.3f}")
print(f"MCC MLP (validation_df): {mcc_val_mlp:.3f}")


MCC MLP (train_df): 0.998
MCC MLP (validation_df): 0.971
